# RAGAS Evaluation: Fireworks AI (OSS) vs OpenAI gpt-4.1-mini

This notebook builds **two RAG pipelines** over the same document corpus (`data/cat-health-guide.pdf`):

| Pipeline | Chat Model | Embedding Model |
|----------|-----------|-----------------|
| **Fireworks (OSS)** | `gpt-oss-20b` via `ChatFireworks` | `nomic-embed-text-v1.5` via Fireworks OpenAI-compat |
| **OpenAI** | `gpt-4.1-mini` via `ChatOpenAI` | `text-embedding-3-small` via `OpenAIEmbeddings` |

We then:
1. Run both pipelines on the same evaluation questions
2. Evaluate with **RAGAS** (Faithfulness, Answer Relevancy, Context Precision)
3. Instrument everything with **LangSmith** for cost & token tracking

## 1 · Environment & API Keys

In [2]:
import os
import getpass
import ssl
from dotenv import load_dotenv

load_dotenv()
ssl._create_default_https_context = ssl._create_unverified_context

# --- Clear any proxy settings ---
for key in ("HTTPS_PROXY", "HTTP_PROXY", "https_proxy", "http_proxy"):
    os.environ[key] = ""

# --- Required API keys ---
if not os.environ.get("FIREWORKS_API_KEY"):
    os.environ["FIREWORKS_API_KEY"] = getpass.getpass("Fireworks API key: ")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "ragas-fireworks-vs-openai"
os.environ["LANGCHAIN_API_KEY"] = os.environ.get("LANGCHAIN_API_KEY")
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

if not os.environ["LANGCHAIN_API_KEY"]:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing disabled")
else:
    print(f"LangSmith tracing enabled. Project: {os.environ['LANGCHAIN_PROJECT']}")

print("✅ Environment configured – LangSmith tracing ON")

LangSmith tracing disabled
✅ Environment configured – LangSmith tracing ON


## 2 · Load & Split Documents

Load the PDF corpus with `langchain_community` and split with `langchain_text_splitters`.

In [3]:
import tiktoken
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

DATA_DIR = "data"

def tiktoken_len(text: str) -> int:
    return len(tiktoken.encoding_for_model("gpt-4o").encode(text))

# Load PDFs
loader = DirectoryLoader(DATA_DIR, glob="**/*.pdf", loader_cls=PyMuPDFLoader)
documents = loader.load()
print(f"📄 Loaded {len(documents)} pages from {DATA_DIR}/")

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=750,
    chunk_overlap=100,
    length_function=tiktoken_len,
)
chunks = text_splitter.split_documents(documents)
print(f"✂️  Split into {len(chunks)} chunks")

📄 Loaded 22 pages from data/
✂️  Split into 44 chunks


## 3 · Build Vector Stores (Fireworks & OpenAI Embeddings)

We create **two separate** Qdrant in-memory vector stores – one per embedding model.

In [5]:
# maknut se s vpn-a i pokrenut

os.environ["HTTPS_PROXY"] = ""
os.environ["HTTP_PROXY"] = ""
os.environ["https_proxy"] = ""
os.environ["http_proxy"] = ""

In [8]:
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

# ── Fireworks Embeddings (via OpenAI-compat endpoint) ──
fireworks_embeddings = OpenAIEmbeddings(
    model="accounts/fireworks/models/qwen3-embedding-8b",
    openai_api_key=os.environ["FIREWORKS_API_KEY"],
    openai_api_base="https://api.fireworks.ai/inference/v1",
    check_embedding_ctx_length=False,
)

fireworks_vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=fireworks_embeddings,
    location=":memory:",
    collection_name="fireworks_rag",
)
fireworks_retriever = fireworks_vectorstore.as_retriever(search_kwargs={"k": 4})
print("🔥 Fireworks vector store ready")

# ── OpenAI Embeddings ──
openai_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

openai_vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=openai_embeddings,
    location=":memory:",
    collection_name="openai_rag",
)
openai_retriever = openai_vectorstore.as_retriever(search_kwargs={"k": 4})
print("🧠 OpenAI vector store ready")

🔥 Fireworks vector store ready
🧠 OpenAI vector store ready


## 4 · Build RAG Pipelines (LangGraph)

Each pipeline is a two-node LangGraph: **retrieve → generate**.

In [9]:
from typing import TypedDict
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_fireworks import ChatFireworks
from langchain_openai import ChatOpenAI
from langgraph.graph import START, StateGraph

# ── Shared RAG state & prompt ──
class RAGState(TypedDict):
    question: str
    context: list[Document]
    response: str

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("human",
     "#CONTEXT:\n{context}\n\n"
     "QUERY:\n{query}\n\n"
     "Use the provided context to answer the query. "
     "Only use the provided context. If the answer is not in the context, say \"I don't know\".")
])


def build_rag_graph(retriever, llm, name: str = "rag"):
    """Build a compiled LangGraph RAG pipeline from a retriever + LLM."""

    def retrieve(state: RAGState) -> RAGState:
        docs = retriever.invoke(state["question"])
        return {"context": docs}

    def generate(state: RAGState) -> RAGState:
        chain = RAG_PROMPT | llm | StrOutputParser()
        ctx_text = "\n\n".join(d.page_content for d in state.get("context", []))
        answer = chain.invoke({"query": state["question"], "context": ctx_text})
        return {"response": answer}

    g = StateGraph(RAGState)
    g = g.add_sequence([retrieve, generate])
    g.add_edge(START, "retrieve")
    return g.compile()


# ── Fireworks RAG (gpt-oss-20b) ──
fireworks_llm = ChatFireworks(
    model="accounts/fireworks/models/gpt-oss-20b",
    temperature=0,
)
fireworks_rag = build_rag_graph(fireworks_retriever, fireworks_llm, "fireworks")
print("🔥 Fireworks RAG pipeline ready")

# ── OpenAI RAG (gpt-4.1-mini) ──
openai_llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
)
openai_rag = build_rag_graph(openai_retriever, openai_llm, "openai")
print("🧠 OpenAI RAG pipeline ready")

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x000001CBD5447110>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x000001CBD5447750>


🔥 Fireworks RAG pipeline ready
🧠 OpenAI RAG pipeline ready


## 5 · Evaluation Questions

Questions about cat health (grounded in the PDF) with reference answers for RAGAS.

In [10]:
eval_questions = [
    "What vaccinations do kittens need and at what age should they start?",
    "How should you manage nutrition for an overweight cat?",
    "What are the common signs of dental disease in cats?",
    "How often should an adult cat visit the veterinarian for a wellness exam?",
    "What are the recommended methods for parasite control in indoor cats?",
    "What behavioral changes may indicate pain or illness in a cat?",
    "What is the recommended diet for senior cats over 10 years old?",
    "How can you tell if a cat is dehydrated and what should you do?",
]

# Reference answers (brief gold-standard answers for RAGAS ground-truth metrics)
eval_references = [
    "Kittens should begin their vaccination series at 6-8 weeks of age, receiving core vaccines including FVRCP (feline viral rhinotracheitis, calicivirus, panleukopenia) and rabies.",
    "Weight management involves controlled portions of a high-protein, low-carb diet, increased play/exercise, and regular veterinary monitoring to avoid hepatic lipidosis from rapid weight loss.",
    "Common signs include bad breath, drooling, difficulty eating, red or swollen gums, pawing at the mouth, and reluctance to eat hard food.",
    "Adult cats should have a veterinary wellness exam at least once a year; senior cats (over 7-10 years) should be seen every 6 months.",
    "Indoor cats should still receive regular flea and intestinal parasite prevention, as parasites can enter on shoes, other pets, or open windows.",
    "Behavioral changes such as hiding, decreased appetite, reduced grooming, aggression when touched, or changes in litter box habits may indicate pain or illness.",
    "Senior cats benefit from easily digestible, high-quality protein diets with controlled phosphorus, added omega-3 fatty acids, and increased moisture content.",
    "Signs of dehydration include skin tenting, dry gums, sunken eyes, and lethargy. Encourage water intake and seek veterinary care if symptoms persist.",
]

print(f"📝 {len(eval_questions)} evaluation questions prepared")

📝 8 evaluation questions prepared


## 6 · Run Both Pipelines

Execute both RAG graphs on every evaluation question and collect answers + retrieved contexts.
All calls are traced to LangSmith automatically.

In [12]:
import time
from langsmith import traceable


@traceable(name="run_single_rag_query")
def run_single_query(rag_graph, question: str, label: str):
    return rag_graph.invoke({"question": question})


def run_pipeline(
    rag_graph,
    retriever,
    questions: list[str],
    label: str,
    max_retries: int = 5,
    base_delay: float = 2.0,
    between_questions_delay: float = 0.5,
):
    """Run a RAG pipeline on all questions and return answers + contexts."""
    answers = []
    contexts = []

    for i, q in enumerate(questions):
        print(f"  [{label}] Q{i+1}/{len(questions)}: {q[:60]}...")

        last_error = None

        for attempt in range(max_retries):
            try:
                result = run_single_query(rag_graph, q, label)

                answers.append(result["response"])
                ctx_texts = [doc.page_content for doc in result.get("context", [])]
                contexts.append(ctx_texts)

                time.sleep(between_questions_delay)
                break

            except Exception as e:
                last_error = e
                err_text = str(e)

                is_rate_limit = (
                    "RATE_LIMIT" in err_text
                    or "RateLimitError" in err_text
                    or "429" in err_text
                )

                if is_rate_limit and attempt < max_retries - 1:
                    sleep_for = base_delay * (2 ** attempt)
                    print(
                        f"    [{label}] Rate limited, retrying in {sleep_for:.1f}s "
                        f"(attempt {attempt + 1}/{max_retries})"
                    )
                    time.sleep(sleep_for)
                else:
                    print(f"    [{label}] Failed: {e}")
                    answers.append(f"ERROR: {e}")
                    contexts.append([])
                    break

        else:
            print(f"    [{label}] Failed after retries: {last_error}")
            answers.append(f"ERROR: {last_error}")
            contexts.append([])

    return answers, contexts


print("─── Running Fireworks RAG pipeline ───")
fw_answers, fw_contexts = run_pipeline(
    fireworks_rag,
    fireworks_retriever,
    eval_questions,
    "Fireworks",
    max_retries=5,
    base_delay=2.0,
    between_questions_delay=1.0,
)

print("\n─── Running OpenAI RAG pipeline ───")
oai_answers, oai_contexts = run_pipeline(
    openai_rag,
    openai_retriever,
    eval_questions,
    "OpenAI",
    max_retries=3,
    base_delay=1.0,
    between_questions_delay=0.3,
)

print("\n✅ Both pipelines finished — check LangSmith for traces!")

─── Running Fireworks RAG pipeline ───
  [Fireworks] Q1/8: What vaccinations do kittens need and at what age should the...
  [Fireworks] Q2/8: How should you manage nutrition for an overweight cat?...
  [Fireworks] Q3/8: What are the common signs of dental disease in cats?...
  [Fireworks] Q4/8: How often should an adult cat visit the veterinarian for a w...
  [Fireworks] Q5/8: What are the recommended methods for parasite control in ind...
  [Fireworks] Q6/8: What behavioral changes may indicate pain or illness in a ca...
  [Fireworks] Q7/8: What is the recommended diet for senior cats over 10 years o...
  [Fireworks] Q8/8: How can you tell if a cat is dehydrated and what should you ...
    [Fireworks] Rate limited, retrying in 2.0s (attempt 1/5)
    [Fireworks] Rate limited, retrying in 4.0s (attempt 2/5)
    [Fireworks] Rate limited, retrying in 8.0s (attempt 3/5)

─── Running OpenAI RAG pipeline ───
  [OpenAI] Q1/8: What vaccinations do kittens need and at what age should the...
  

## 7 · Preview Answers Side-by-Side

In [13]:
import pandas as pd

preview_df = pd.DataFrame({
    "Question": eval_questions,
    "Fireworks Answer": [a[:200] + "..." if len(a) > 200 else a for a in fw_answers],
    "OpenAI Answer":    [a[:200] + "..." if len(a) > 200 else a for a in oai_answers],
})
preview_df

,Question,Fireworks Answer,OpenAI Answer
0,What vaccinations do kittens need and at what ...,**Core vaccines for kittens**\n\n| Vaccine | W...,Kittens need core vaccines including rabies vi...
1,How should you manage nutrition for an overwei...,**Managing nutrition for an overweight cat**\n...,"To manage nutrition for an overweight cat, the..."
2,What are the common signs of dental disease in...,I don't know.,Common signs of dental disease in cats include...
3,How often should an adult cat visit the veteri...,Adult cats should have a minimum of an annual ...,An adult cat should have a minimum of annual e...
4,What are the recommended methods for parasite ...,**Recommended parasite‑control methods for ind...,The recommended methods for parasite control i...
5,What behavioral changes may indicate pain or i...,Behavioral changes that may signal pain or ill...,Behavioral changes that may indicate pain or i...
6,What is the recommended diet for senior cats o...,**Recommended diet for senior cats ( > 10 year...,"For senior cats over 10 years old, the resting..."
7,How can you tell if a cat is dehydrated and wh...,I don't know.,The provided context does not include specific...


## 8 · RAGAS Evaluation

Evaluate both pipelines with RAGAS metrics:
- **Faithfulness** – Is the answer grounded in the retrieved context?
- **Answer Relevancy** – Is the answer relevant to the question?
- **Context Precision** – Are the retrieved chunks relevant to answering the question?

> RAGAS uses an LLM judge internally. We use OpenAI `gpt-4.1-mini` as the evaluator for both pipelines (fair comparison).

In [18]:
from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini", temperature=0))
evaluator_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model="text-embedding-3-small")
)

metrics = [
    Faithfulness(llm=evaluator_llm),
    AnswerRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
    ContextPrecision(llm=evaluator_llm),
]

def build_ragas_dataset(questions, answers, contexts, references):
    samples = []
    for q, a, ctx, ref in zip(questions, answers, contexts, references):
        samples.append(
            SingleTurnSample(
                user_input=q,
                response=a,
                retrieved_contexts=ctx,
                reference=ref,
            )
        )
    return EvaluationDataset(samples=samples)

fw_dataset = build_ragas_dataset(eval_questions, fw_answers, fw_contexts, eval_references)
oai_dataset = build_ragas_dataset(eval_questions, oai_answers, oai_contexts, eval_references)

fw_result = evaluate(dataset=fw_dataset, metrics=metrics)
oai_result = evaluate(dataset=oai_dataset, metrics=metrics)

print("Fireworks:", fw_result)
print("OpenAI:", oai_result)

C:\Users\dznidaric\AppData\Local\Temp\ipykernel_49312\3483064409.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision
C:\Users\dznidaric\AppData\Local\Temp\ipykernel_49312\3483064409.py:2: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision
C:\Users\dznidaric\AppData\Local\Temp\ipykernel_49312\3483064409.py:2: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections 

Evaluating:   0%|          | 0/24 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Evaluating:   0%|          | 0/24 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Fireworks: {'faithfulness': 0.6380, 'answer_relevancy': 0.6980, 'context_precision': 0.6146}
OpenAI: {'faithfulness': 0.9617, 'answer_relevancy': 0.6917, 'context_precision': 0.5625}


In [19]:
print("─── Evaluating Fireworks pipeline ───")
fw_results = evaluate(dataset=fw_dataset, metrics=metrics)

print("\n─── Evaluating OpenAI pipeline ───")
oai_results = evaluate(dataset=oai_dataset, metrics=metrics)

print("\n✅ RAGAS evaluation complete!")

─── Evaluating Fireworks pipeline ───


Evaluating:   0%|          | 0/24 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.



─── Evaluating OpenAI pipeline ───


Evaluating:   0%|          | 0/24 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.



✅ RAGAS evaluation complete!


## 9 · Compare RAGAS Results

In [20]:
# ── Per-question detail ──
fw_detail = fw_results.to_pandas()
fw_detail.insert(0, "pipeline", "Fireworks (gpt-oss-20b)")

oai_detail = oai_results.to_pandas()
oai_detail.insert(0, "pipeline", "OpenAI (gpt-4.1-mini)")

all_detail = pd.concat([fw_detail, oai_detail], ignore_index=True)
all_detail

,pipeline,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision
0,Fireworks (gpt-oss-20b),What vaccinations do kittens need and at what ...,[as transport hosts of infected ticks to human...,**Core vaccines for kittens**\n\n| Vaccine | W...,Kittens should begin their vaccination series ...,0.629630,0.744560,1.000000
1,Fireworks (gpt-oss-20b),How should you manage nutrition for an overwei...,[have a very high energy requirement of 200 kc...,**Managing nutrition for an overweight cat**\n...,Weight management involves controlled portions...,0.725490,0.952636,0.000000
2,Fireworks (gpt-oss-20b),What are the common signs of dental disease in...,[detection of changes and identiﬁcation of tre...,I don't know.,"Common signs include bad breath, drooling, dif...",0.000000,0.000000,0.000000
3,Fireworks (gpt-oss-20b),How often should an adult cat visit the veteri...,"[For example, some senior cats aged 10 years a...",Adult cats should have a minimum of an annual ...,Adult cats should have a veterinary wellness e...,1.000000,0.937587,1.000000
4,Fireworks (gpt-oss-20b),What are the recommended methods for parasite ...,"[disease, masses, or orofacial pain can be mad...",**Recommended parasite‑control methods for ind...,Indoor cats should still receive regular flea ...,0.611111,0.975020,1.000000
5,Fireworks (gpt-oss-20b),What behavioral changes may indicate pain or i...,[Detecting signs of pain or anxiety and evalua...,Behavioral changes that may signal pain or ill...,"Behavioral changes such as hiding, decreased a...",1.000000,0.973827,1.000000
6,Fireworks (gpt-oss-20b),What is the recommended diet for senior cats o...,[good starting point is to calculate the adult...,**Recommended diet for senior cats ( > 10 year...,"Senior cats benefit from easily digestible, hi...",1.000000,1.000000,1.000000
7,Fireworks (gpt-oss-20b),How can you tell if a cat is dehydrated and wh...,"[L-carnitine, antioxidants, and amino acids to...",I don't know.,"Signs of dehydration include skin tenting, dry...",0.000000,0.000000,0.000000
8,OpenAI (gpt-4.1-mini),What vaccinations do kittens need and at what ...,[may even use tactics to encourage comfort suc...,Kittens need core vaccines including rabies vi...,Kittens should begin their vaccination series ...,1.000000,0.760862,0.333333
9,OpenAI (gpt-4.1-mini),How should you manage nutrition for an overwei...,[have a very high energy requirement of 200 kc...,"To manage nutrition for an overweight cat, the...",Weight management involves controlled portions...,0.904762,0.973566,0.500000


In [21]:
# ── Aggregate comparison ──
metric_cols = [c for c in fw_detail.columns if c not in ("pipeline", "user_input", "response", "retrieved_contexts", "reference")]

summary = pd.DataFrame({
    "Metric": metric_cols,
    "Fireworks (gpt-oss-20b)": [fw_detail[c].mean() for c in metric_cols],
    "OpenAI (gpt-4.1-mini)":   [oai_detail[c].mean() for c in metric_cols],
})
summary["Δ (OpenAI − Fireworks)"] = summary["OpenAI (gpt-4.1-mini)"] - summary["Fireworks (gpt-oss-20b)"]
summary.style.format("{:.4f}", subset=["Fireworks (gpt-oss-20b)", "OpenAI (gpt-4.1-mini)", "Δ (OpenAI − Fireworks)"])

,Metric,Fireworks (gpt-oss-20b),OpenAI (gpt-4.1-mini),Δ (OpenAI − Fireworks)
0,faithfulness,0.6208,0.9617,0.3409
1,answer_relevancy,0.6980,0.6943,-0.0037
2,context_precision,0.6250,0.5937,-0.0313


## 10 · Cost & Token Analysis (LangSmith)

Use the **LangSmith** Python client to pull token usage and estimated cost from the traces we just generated.

> Open [smith.langchain.com](https://smith.langchain.com) → project **ragas-fireworks-vs-openai** to see the full cost dashboard.

In [22]:
from langsmith import Client

ls_client = Client()
project_name = "ragas-fireworks-vs-openai"

# Pull all runs from our project
runs = list(ls_client.list_runs(
    project_name=project_name,
    run_type="llm",
))

# Separate by provider
fireworks_runs = [r for r in runs if r.extra and "fireworks" in str(r.extra.get("invocation_params", {}).get("model", "")).lower()]
openai_runs = [r for r in runs if r.extra and "gpt-4" in str(r.extra.get("invocation_params", {}).get("model", "")).lower()]

def summarize_runs(run_list, label):
    total_prompt = sum((r.prompt_tokens or 0) for r in run_list)
    total_completion = sum((r.completion_tokens or 0) for r in run_list)
    total_cost = sum((r.total_cost or 0) for r in run_list)
    return {
        "Pipeline": label,
        "LLM Calls": len(run_list),
        "Prompt Tokens": total_prompt,
        "Completion Tokens": total_completion,
        "Total Tokens": total_prompt + total_completion,
        "Total Cost ($)": f"${total_cost:.6f}",
    }

cost_df = pd.DataFrame([
    summarize_runs(fireworks_runs, "Fireworks (gpt-oss-20b)"),
    summarize_runs(openai_runs, "OpenAI (gpt-4.1-mini)"),
])
cost_df

LangSmithNotFoundError: Project ragas-fireworks-vs-openai not found

## 11 · Summary & Analysis

In [23]:
print("=" * 70)
print("RAGAS EVALUATION SUMMARY")
print("=" * 70)
print()
print(summary.to_string(index=False))
print()
print("=" * 70)
print("COST & TOKEN USAGE (from LangSmith)")
print("=" * 70)
print()
print(cost_df.to_string(index=False))
print()
print("─" * 70)
print("Key takeaways:")
print("  • Compare Faithfulness scores: higher = answer better grounded in context")
print("  • Compare Response Relevancy:  higher = more relevant to user question")
print("  • Compare Context Precision:   higher = retriever found better chunks")
print("  • Check LangSmith dashboard for detailed per-query cost breakdowns:")
print("    → https://smith.langchain.com  (project: ragas-fireworks-vs-openai)")
print("─" * 70)

RAGAS EVALUATION SUMMARY

           Metric  Fireworks (gpt-oss-20b)  OpenAI (gpt-4.1-mini)  Δ (OpenAI − Fireworks)
     faithfulness                 0.620779               0.961706                0.340927
 answer_relevancy                 0.697954               0.694294               -0.003660
context_precision                 0.625000               0.593750               -0.031250

COST & TOKEN USAGE (from LangSmith)



NameError: name 'cost_df' is not defined